In [13]:
import pandas as pd
import requests
from tqdm import tqdm
import time
from Bio import AlignIO
import numpy as np
from collections import Counter
import math

In [4]:
import pandas as pd
import numpy as np

mutation_pool = pd.read_csv("../mutation_pool_all.csv")

mpnn = pd.read_csv(
    "../../MPNN/ThermoMPNN_pool.csv"
)

print("mutation pool:", mutation_pool.shape)
print("MPNN:", mpnn.shape)

mutation_pool.head()

mutation pool: (1451, 16)
MPNN: (5662, 5)


,Position,WT,Mut,Delta_logP,Rank,Alignment_Pos,Entropy,Top_mutations,Source,BLOSUM,BioBonus,ESM_norm,Entropy_norm,BLOSUM_norm,BioBonus_norm,ProxyScore
0,176,Y,L,8.613548,1,6554,0.821826,"['L', 'F', 'M', 'I', 'V']",2,-1.0,2,1.000000,0.207189,0.428571,1.0,0.755723
1,138,M,V,7.391602,2,6187,1.654679,"['V', 'A', 'I', 'L', 'S']",2,1.0,0,0.946297,0.428482,0.714286,0.0,0.665988
2,140,K,S,2.751953,31,6227,3.662069,"['S', 'T', 'M', 'A', 'R']",0,0.0,0,0.742389,0.961854,0.571429,0.0,0.649279
3,113,S,R,-2.199219,874,5526,3.283589,"['Y', 'R', 'T', 'A', 'V']",0,-1.0,2,0.524790,0.861290,0.428571,1.0,0.648939
4,208,L,Q,4.635254,7,6976,3.641430,"['Q', 'E', 'A', 'K', 'P']",0,-2.0,0,0.825158,0.956370,0.285714,0.0,0.646710


In [5]:
mpnn.head()

,position,WT,Mut,ThermoMPNN_ddG,ThermoMPNN_norm
0,1,M,A,-0.032745,0.687929
1,1,M,C,-0.214785,0.716352
2,1,M,D,-0.002282,0.683173
3,1,M,E,0.055492,0.674152
4,1,M,F,-0.005874,0.683733


In [6]:
merged = mutation_pool.merge(
    mpnn,
    left_on=["Position","WT","Mut"],
    right_on=["position","WT","Mut"],
    how="left"
)

merged.head()

,Position,WT,Mut,Delta_logP,Rank,Alignment_Pos,Entropy,Top_mutations,Source,BLOSUM,BioBonus,ESM_norm,Entropy_norm,BLOSUM_norm,BioBonus_norm,ProxyScore,position,ThermoMPNN_ddG,ThermoMPNN_norm
0,176,Y,L,8.613548,1,6554,0.821826,"['L', 'F', 'M', 'I', 'V']",2,-1.0,2,1.000000,0.207189,0.428571,1.0,0.755723,176,1.631778,0.428036
1,138,M,V,7.391602,2,6187,1.654679,"['V', 'A', 'I', 'L', 'S']",2,1.0,0,0.946297,0.428482,0.714286,0.0,0.665988,138,1.328275,0.475424
2,140,K,S,2.751953,31,6227,3.662069,"['S', 'T', 'M', 'A', 'R']",0,0.0,0,0.742389,0.961854,0.571429,0.0,0.649279,140,1.335361,0.474317
3,113,S,R,-2.199219,874,5526,3.283589,"['Y', 'R', 'T', 'A', 'V']",0,-1.0,2,0.524790,0.861290,0.428571,1.0,0.648939,113,0.122951,0.663619
4,208,L,Q,4.635254,7,6976,3.641430,"['Q', 'E', 'A', 'K', 'P']",0,-2.0,0,0.825158,0.956370,0.285714,0.0,0.646710,208,0.093802,0.668170


In [7]:
print(
    "MPNN matched:",
    merged["ThermoMPNN_ddG"].notna().sum(),
    "/",
    len(merged)
)

MPNN matched: 1451 / 1451


In [8]:
merged.drop(
    columns=["position"],
    inplace=True
)

In [15]:
print(merged.head())

   Position WT Mut  Delta_logP  Rank  Alignment_Pos   Entropy  \
0       176  Y   L    8.613548     1           6554  0.821826   
1       138  M   V    7.391602     2           6187  1.654679   
2       140  K   S    2.751953    31           6227  3.662069   
3       113  S   R   -2.199219   874           5526  3.283589   
4       208  L   Q    4.635254     7           6976  3.641430   

               Top_mutations  Source  BLOSUM  BioBonus  ESM_norm  \
0  ['L', 'F', 'M', 'I', 'V']       2    -1.0         2  1.000000   
1  ['V', 'A', 'I', 'L', 'S']       2     1.0         0  0.946297   
2  ['S', 'T', 'M', 'A', 'R']       0     0.0         0  0.742389   
3  ['Y', 'R', 'T', 'A', 'V']       0    -1.0         2  0.524790   
4  ['Q', 'E', 'A', 'K', 'P']       0    -2.0         0  0.825158   

   Entropy_norm  BLOSUM_norm  BioBonus_norm  ProxyScore  ThermoMPNN_ddG  \
0      0.207189     0.428571            1.0    0.755723        1.631778   
1      0.428482     0.714286            0.0    0.6

In [16]:
merged["ProxyScore_MPNN"] = (
    0.35*merged["ESM_norm"]
    +
    0.15*merged["Entropy_norm"]
    +
    0.15*merged["BLOSUM_norm"]
    +
    0.1*merged["BioBonus_norm"]
    +
    0.25*merged["ThermoMPNN_norm"]
)

In [17]:
merged.sort_values(
    "ProxyScore_MPNN",
    ascending=False
).head(20)

,Position,WT,Mut,Delta_logP,Rank,Alignment_Pos,Entropy,Top_mutations,Source,BLOSUM,BioBonus,ESM_norm,Entropy_norm,BLOSUM_norm,BioBonus_norm,ProxyScore,ThermoMPNN_ddG,ThermoMPNN_norm,ProxyScore_MPNN
15,103,E,K,3.009521,24,5422,2.535734,"['K', 'R', 'Q', 'P', 'N']",2,1.0,0,0.753709,0.662582,0.714286,0.0,0.616514,-1.070972,0.850035,0.682837
65,147,E,K,-0.638672,502,6411,3.537182,"['K', 'L', 'R', 'N', 'S']",0,1.0,0,0.593374,0.928671,0.714286,0.0,0.589564,-1.462576,0.911178,0.681919
25,40,D,E,0.154297,292,4598,3.147583,"['E', 'K', 'V', 'T', 'Q']",0,2.0,0,0.628224,0.825152,0.857143,0.0,0.607714,-0.853618,0.816098,0.676247
6,89,Y,F,-0.082031,341,5156,3.362808,"['L', 'F', 'M', 'V', 'I']",0,3.0,0,0.617838,0.882339,1.000000,0.0,0.635387,0.092379,0.668392,0.665692
137,40,D,T,1.263672,104,4598,3.147583,"['E', 'K', 'V', 'T', 'Q']",0,-1.0,0,0.676980,0.825152,0.428571,0.0,0.567806,-1.711517,0.950047,0.662514
17,158,E,Q,-0.187500,370,6442,3.419280,"['K', 'Q', 'A', 'R', 'S']",0,2.0,0,0.613203,0.897344,0.857143,0.0,0.614642,-0.338071,0.735602,0.661694
16,270,V,I,-1.046875,607,7807,3.401873,"['D', 'I', 'Q', 'A', 'E']",0,3.0,0,0.575434,0.892719,1.000000,0.0,0.616261,-0.120048,0.701560,0.660700
12,161,V,I,-0.292969,408,6456,3.168278,"['L', 'R', 'K', 'I', 'A']",0,3.0,0,0.608568,0.830651,1.000000,0.0,0.620414,-0.059655,0.692131,0.660629
27,212,D,E,-0.148438,358,6995,3.243826,"['E', 'N', 'K', 'R', 'S']",0,2.0,0,0.614920,0.850725,0.857143,0.0,0.606176,-0.455610,0.753954,0.659890
61,212,D,N,0.109375,301,6995,3.243826,"['E', 'N', 'K', 'R', 'S']",0,1.0,0,0.626250,0.850725,0.714286,0.0,0.590413,-0.823098,0.811332,0.656772


In [12]:
merged.to_csv(
    "../mutation_pool_MPNN.csv",
    index=False
)